Set up agent

In [ ]:
"""
This script provides a self-contained, single-file implementation of a FEMA (Federal
Emergency Management Agency) assistant agent.

Functionality:
1.  Dependency Installation: Automatically installs all required Python packages,
    including LangChain and Google Cloud libraries, using 'uv' for fast
    installation.
2.  Agent Initialization: Sets up a conversational agent using the ReAct
    (Reasoning and Acting) framework with LangChain.
3.  Tool Integration: Equips the agent with a Google Search tool to access
    real-time information. This requires setting the GOOGLE_API_KEY and
    GOOGLE_CSE_ID environment variables.
4.  LLM Integration: Utilizes Google's Vertex AI 'gemini-pro' model as the
    reasoning engine for the agent.
5.  Interactive Prompt: Runs a command-line loop where a user can ask
    questions related to FEMA. The agent will show its thought process and
    actions as it arrives at a final answer.

Usage:
    1.  (Optional) Set Google Search API credentials:
        export GOOGLE_API_KEY="YOUR_API_KEY"
        export GOOGLE_CSE_ID="YOUR_SEARCH_ENGINE_ID"
    2.  Run the script from the command line:
        python3 fema_agent_onescript.py
"""
import subprocess
import sys
import os

def install_dependencies():
    """Installs required Python packages using uv."""
    packages = [
        "google-cloud-aiplatform",
        "google-cloud-discoveryengine",
        "google-api-python-client",
        "langchain-google-vertexai",
        "google-cloud-logging",
        "langchain",
        "langchain-core",
        "langchain-community",
        "google-cloud-storage",
        "google-auth",
        "google-auth-oauthlib",
        "google-auth-httplib2",
        "uv", # Ensure uv is available
    ]
    print("--- Installing dependencies ---")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "uv"])
        subprocess.check_call([sys.executable, "-m", "uv", "pip", "install"] + packages)
        print("--- Dependencies installed successfully ---")
    except subprocess.CalledProcessError as e:
        print(f"Error during dependency installation: {e}", file=sys.stderr)
        sys.exit(1)

def run_agent():
    """
    Initializes and runs the FEMA agent.
    """
    print("\n--- Initializing FEMA Agent ---")

    # Ensure dependencies are loaded in the current process
    try:
        from langchain_google_vertexai import VertexAI
        from langchain.agents import AgentExecutor, create_react_agent
        from langchain_core.prompts import PromptTemplate
        from langchain_community.tools import GoogleSearchRun
        from langchain_community.utilities import GoogleSearchAPIWrapper
    except ImportError:
        print("Could not import necessary libraries. Please ensure installation was successful.", file=sys.stderr)
        sys.exit(1)

    # --- Agent Definition ---

    # 1. Define Tools
    # The agent will use Google Search to find information.
    # NOTE: This requires Google Search API credentials to be set up.
    # See: https://python.langchain.com/v0.2/docs/integrations/tools/google_search/
    # You need to set GOOGLE_CSE_ID and GOOGLE_API_KEY environment variables.
    try:
        if "GOOGLE_API_KEY" not in os.environ or "GOOGLE_CSE_ID" not in os.environ:
            print("\nWARNING: GOOGLE_API_KEY and/or GOOGLE_CSE_ID environment variables not set.")
            print("The agent's search tool may not function correctly.")
            print("See: https://python.langchain.com/v0.2/docs/integrations/tools/google_search/")

        search = GoogleSearchAPIWrapper()
        search_tool = GoogleSearchRun(api_wrapper=search, name="google_search")
        tools = [search_tool]
    except Exception as e:
        print(f"Error initializing Google Search tool: {e}", file=sys.stderr)
        print("The agent will proceed without the search tool.", file=sys.stderr)
        tools = []

    # 2. Define the LLM
    # Using Vertex AI's Gemini Pro model.
    llm = VertexAI(model_name="gemini-pro")

    # 3. Define the Prompt Template
    # This template guides the agent's reasoning process (ReAct framework).
    template = """
    You are an agent designed to assist with inquiries related to the US Federal Emergency Management Agency (FEMA).
    Answer the following questions as best you can. You have access to the following tools:

    {tools}

    Use the following format:

    Question: the input question you must answer
    Thought: you should always think about what to do
    Action: the action to take, should be one of [{tool_names}]
    Action Input: the input to the action
    Observation: the result of the action
    ... (this Thought/Action/Action Input/Observation can repeat N times)
    Thought: I now know the final answer
    Final Answer: the final answer to the original input question

    Begin!

    Question: {input}
    Thought:{agent_scratchpad}
    """
    prompt = PromptTemplate.from_template(template)

    # 4. Create the Agent
    agent = create_react_agent(llm, tools, prompt)
    agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

    # --- Run the Agent ---
    print("\n--- FEMA Agent is ready. Ask a question. (Type 'exit' to quit) ---")
    while True:
        query = input("Question: ")
        if query.lower() == 'exit':
            break
        agent_executor.invoke({"input": query})

if __name__ == "__main__":
    install_dependencies()
    run_agent()

Deploy

In [ ]:
"""
This script provides a self-contained, single-file implementation to define and
deploy a FEMA (Federal Emergency Management Agency) assistant agent to Google
Cloud's Vertex AI Agent Engine.

Functionality:
1.  Dependency Installation: Automatically installs all required Python packages,
    including the Google Agent Development Kit (ADK) and Vertex AI libraries.
2.  Agent Definition (ADK): Defines a FEMA agent using the `google-adk`
    framework. This includes defining tools, models, and instructions for the
    agent's behavior.
3.  Tool Integration: Equips the agent with a `CodeTool` for executing
    Python functions. A placeholder `search_fema_database` tool is provided
    as an example.
4.  Deployment Logic: Contains the function to package the ADK agent and
    deploy it to Vertex AI Agent Engine, specifying the necessary project ID,
    location, and display name.

Usage:
    1.  Ensure you are authenticated with Google Cloud:
        gcloud auth application-default login

    2.  Set your Google Cloud Project ID in the `deploy_agent` function.

    3.  Run the script from the command line:
        python3 deploy_fema_agent_onescript.py
"""
import subprocess
import sys
import os

def install_dependencies():
    """Installs required Python packages for ADK and deployment."""
    # The 'google-adk' library is in pre-release.
    packages = [
        "google-cloud-aiplatform[adk,agent_engines]",
        "google-adk",
    ]
    print("--- Installing deployment dependencies ---")
    try:
        # Using pip directly as uv is not a dependency for this script's logic
        subprocess.check_call([sys.executable, "-m", "pip", "install"] + packages)
        print("--- Dependencies installed successfully ---")
    except subprocess.CalledProcessError as e:
        print(f"Error during dependency installation: {e}", file=sys.stderr)
        sys.exit(1)

def define_and_deploy_agent():
    """
    Defines the FEMA agent using the Google ADK and deploys it to
    Vertex AI Agent Engine.
    """
    print("\n--- Defining and Deploying FEMA Agent ---")

    try:
        from google.adk.agents import Agent
        from google.adk.apps import App
        from google.adk.models import Gemini
        from google.adk.tools import CodeTool
        from vertexai import agent_engines
    except ImportError:
        print("Could not import ADK/Vertex AI libraries. Please ensure installation was successful.", file=sys.stderr)
        sys.exit(1)

    # --- Agent Definition (using google-adk) ---

    # 1. Define a Tool
    # This is a placeholder tool. In a real-world scenario, this could
    # connect to a FEMA database, API, or other data source.
    def search_fema_database(query: str) -> str:
        """Searches the FEMA database for information on a given query."""
        # This is a mock implementation.
        if "hurricane" in query.lower():
            return "FEMA is actively monitoring Hurricane Zeta. Resources are being deployed to affected areas."
        elif "assistance" in query.lower():
            return "To apply for disaster assistance, please visit disasterassistance.gov."
        return f"No specific information found for '{query}'. General information is available at fema.gov."

    # 2. Define the Agent
    fema_agent = Agent(
        model=Gemini(model="gemini-1.5-flash-001"),
        instruction="You are a helpful FEMA assistant. Use your tools to answer questions about disaster relief, preparedness, and official FEMA information.",
        tools=[CodeTool(search_fema_database)],
    )

    # 3. Create the App
    # The App object is the deployable unit for Agent Engine.
    agent_app = App(root_agent=fema_agent, name="fema-agent-app")

    # --- Deployment Logic ---
    project_id = "qwiklabs-gcp-01-0c244accce49"  # <--- IMPORTANT: SET YOUR GCP PROJECT ID HERE
    location = "us-central1"

    print(f"\nDeploying agent to project '{project_id}' in '{location}'...")

    remote_app = agent_engines.create(
        agent_app,
        display_name="FEMA Assistant Agent",
    )

    print("\n--- Agent Deployed Successfully! ---")
    print(f"  Resource Name: {remote_app.resource_name}")
    print(f"  Display Name: {remote_app.display_name}")

if __name__ == "__main__":
    install_dependencies()
    define_and_deploy_agent()